# Chapter 3: Metric Products of Subspaces

This is a standalone study notebook for the printed Chapter 3 span, pp. 65-98. It is original prose and executable material: the aim is to rebuild the chapter's ideas in a notebook-native form, with numerical invariants and interactive figures instead of copied textbook exposition.

Chapter 2 gave us the outer product, so we could make oriented subspaces. Chapter 3 adds the metric layer: once vectors know how to take dot products, blades can know their sizes, relative attitudes, perpendicular complements, projections, and reciprocal coordinate systems.


In [1]:
from pathlib import Path
import sys

import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "ga").exists():
        BOOK_ROOT = candidate
        break

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_plotly_html
from utils.ga import Algebra
from utils.chapter03_metric_products import (
    blade_cosine,
    blade_measure,
    blade_metric_product,
    blade_norm_squared,
    color_space_convert,
    color_space_figure,
    contraction_figure,
    coordinates_in_frame,
    cross_from_dual_wedge,
    gram_matrix,
    project_vector_to_span,
    projection_figure,
    reciprocal_frame,
    reciprocal_frame_figure,
    sample_rgb_points,
    scalar_product_star,
    vector_angle,
    vector_norm,
)

np.set_printoptions(precision=4, suppress=True)
ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
ARTIFACT_TOPIC = "chapter-03"
print(f"project root: {BOOK_ROOT}")
print(f"artifact root: {ARTIFACT_ROOT / ARTIFACT_TOPIC}")


project root: Geometric-Algebra-for-Computer-Science
artifact root: Geometric-Algebra-for-Computer-Science/artifacts/chapter-03


## Source Span and Headings

I verified the page span against the local PDF before authoring this notebook. PDF page 97 is printed page 65, where Chapter 3 begins, and PDF page 130 is printed page 98, after the color-space conversion example. Printed page 99 begins Chapter 4.

The headings covered here are the chapter title, sizing subspaces, scalar product, squared norms, subspace angles, contraction, right contraction, orthogonality and duality, projection, the 3-D cross product, reciprocal frames, further reading, exercises, and programming examples through color-space conversion.


In [2]:
page_audit = {
    "printed_pages": "65-98",
    "pdf_pages": "97-130",
    "start_heading": "3 Metric Products of Subspaces",
    "end_heading": "3.11.4 Color Space Conversion",
    "next_printed_page": "99 begins Chapter 4: Linear Transformations of Subspaces",
    "covered_headings": [
        "3.1 Sizing Up Subspaces",
        "3.1.1 Metrics, Norms, and Angles",
        "3.1.2 Definition of the Scalar Product",
        "3.1.3 The Squared Norm of a Subspace",
        "3.1.4 The Angle Between Subspaces",
        "3.2 From Scalar Product to Contraction",
        "3.3 Geometric Interpretation of the Contraction",
        "3.4 The Other Contraction",
        "3.5 Orthogonality and Duality",
        "3.6 Orthogonal Projection of Subspaces",
        "3.7 The 3-D Cross Product",
        "3.8 Application: Reciprocal Frames",
        "3.11 Programming Examples and Exercises",
    ],
}
audit_path = save_json(page_audit, ARTIFACT_TOPIC, "checks", "source-page-audit.json", root=ARTIFACT_ROOT)
page_audit, audit_path


({'printed_pages': '65-98',
  'pdf_pages': '97-130',
  'start_heading': '3 Metric Products of Subspaces',
  'end_heading': '3.11.4 Color Space Conversion',
  'next_printed_page': '99 begins Chapter 4: Linear Transformations of Subspaces',
  'covered_headings': ['3.1 Sizing Up Subspaces',
   '3.1.1 Metrics, Norms, and Angles',
   '3.1.2 Definition of the Scalar Product',
   '3.1.3 The Squared Norm of a Subspace',
   '3.1.4 The Angle Between Subspaces',
   '3.2 From Scalar Product to Contraction',
   '3.3 Geometric Interpretation of the Contraction',
   '3.4 The Other Contraction',
   '3.5 Orthogonality and Duality',
   '3.6 Orthogonal Projection of Subspaces',
   '3.7 The 3-D Cross Product',
   '3.8 Application: Reciprocal Frames',
   '3.11 Programming Examples and Exercises']},
 WindowsPath('Geometric-Algebra-for-Computer-Science/artifacts/chapter-03/checks/source-page-audit.json'))

## Chapter Roadmap

The chapter is best read as a steady enrichment of the subspace algebra from Chapter 2. The outer product already knows how to build a line, plane, or volume from independent directions. It deliberately ignores metric information: parallel shear does not change an outer-product area, and no dot product is needed to say that a blade has an attitude and an orientation. That nonmetric freedom is powerful, but it leaves common questions unanswered. A graphics program eventually needs to know whether two surface patches are nearly parallel, how far a vector sticks out of a plane, or what coordinates a color has in a skewed set of primaries. Those are metric questions.

The first step is to extend vector dot products to same-grade blades. For vectors this is familiar length and angle. For bivectors and trivectors the determinant form says that the comparison must account for every spanning direction at once. This is why a Gram determinant appears whenever we measure an area or a volume. If the spanning factors become dependent, the determinant collapses to zero, just as the blade itself loses dimension.

The second step is contraction. Scalar products compare equal grades, but geometry constantly asks unequal-grade questions: how does a vector meet a plane, what part of a bivector remains after one direction has been tested, and how should a higher-grade blade encode perpendicularity? Contraction answers by lowering grade. It is metric because it uses dot products internally; it is still geometric because the result is a blade-like element, not a loose collection of components.

The third step is duality. Once a full-space pseudoscalar is available, every direct subspace has an orthogonal-complement description. A plane in 3-D can be handled as the plane itself or as its normal vector. A line can be handled as a direction or as the plane perpendicular to it. The notebook keeps both views close together because practical code often moves between them: direct blades are natural for spans and incidence, while dual blades are natural for normals, constraints, and projection formulas.

The last step is application. Projection, cross products, reciprocal frames, and color-space conversion are not separate tricks; they are examples of the same metric products doing useful work. The saved figures are meant to make this explicit. If you change the input vectors, the algebra should continue to preserve the stated invariant: areas come from determinants, residuals are orthogonal, reciprocal frames produce a Kronecker-delta table, and color coordinates reconstruct the original vector before clipping.


## Metric Data for Blades

A dot product tells vectors how to compare lengths and directions. For blades, the same metric spreads across every spanning vector. The resulting determinant is a compact way to ask: how much of one oriented k-dimensional element matches another k-dimensional element?

One subtle sign convention is worth making explicit. The chapter's scalar-product convention behaves like the scalar part of a geometric product. In Euclidean 3-D, a unit bivector such as `e1 wedge e2` has scalar product `-1` with itself under that convention. The positive area measure comes from comparing a blade with its reverse-adjusted partner. That is why the helper module keeps both `scalar_product_star` and `blade_metric_product` visible.


In [3]:
e1 = np.array([1.0, 0.0, 0.0])
e2 = np.array([0.0, 1.0, 0.0])
e3 = np.array([0.0, 0.0, 1.0])

A = np.array([e1, e2])
tilted = np.array([e1, 0.45 * e2 + e3])
stretched = np.array([2.0 * e1, -0.25 * e1 + 1.4 * e2])

metric_report = {
    "vector_angle_e1_to_tilted_second_degrees": float(np.degrees(vector_angle(e1, tilted[1]))),
    "gram_A": gram_matrix(A).round(6).tolist(),
    "star_A_A": scalar_product_star(A, A),
    "metric_product_A_A": blade_metric_product(A, A),
    "area_A": blade_measure(A),
    "area_stretched": blade_measure(stretched),
    "plane_cosine_A_tilted": blade_cosine(A, tilted),
    "tilted_area": blade_measure(tilted),
}

assert metric_report["star_A_A"] == -1.0
assert abs(metric_report["metric_product_A_A"] - 1.0) < 1e-12
assert abs(metric_report["area_A"] - 1.0) < 1e-12
metric_report


{'vector_angle_e1_to_tilted_second_degrees': 90.0,
 'gram_A': [[1.0, 0.0], [0.0, 1.0]],
 'star_A_A': -1.0,
 'metric_product_A_A': 1.0,
 'area_A': 1.0,
 'area_stretched': 2.8,
 'plane_cosine_A_tilted': 0.41036467732879783,
 'tilted_area': 1.0965856099730655}

## Reading the Determinant Geometrically

The determinant in the scalar product should not be read as a return to coordinate bookkeeping. It is a compact test for oriented content. For two vectors, the comparison matrix has one entry, so the scalar product is the dot product. For two bivectors, the comparison matrix records all pairwise dot products between one pair of spanning vectors and the other pair. The determinant then keeps the alternating behavior that made the outer product meaningful in the first place. Swapping two factors reverses orientation, and linearly dependent factors remove the area.

This point matters in implementations. It is tempting to store a bivector as three numbers in 3-D and then compare those numbers with an ordinary vector dot product. That can work in a carefully chosen Euclidean basis, but it hides the reason the computation is valid and makes later generalization harder. The determinant version states the invariant directly: the blade comparison is induced by the metric on the original vector space. If the metric changes, or if the frame is not orthonormal, the same function still says what must be recomputed.

The sign convention also deserves attention. A blade carries orientation, and the scalar part of a product remembers the algebraic ordering used to build it. For even-grade blades in Euclidean space, a blade multiplied by itself can be negative. This is not a contradiction; it is a reminder that measurement and multiplication are related but not identical. The positive size is obtained by using the reverse-adjusted comparison. In the code, `scalar_product_star(A, A)` preserves the product convention, while `blade_metric_product(A, A)` gives the positive metric measure used by norms and angle checks.

A useful mental check is the unit square in the `e1,e2` plane. Its oriented area blade is not a scalar; it is the area element `e1 wedge e2`. The star product of that bivector with itself reports the orientation-sensitive product sign. The reverse-adjusted metric product reports area squared. Both are meaningful, and confusing them is one of the fastest ways to create sign errors in contraction and projection code.


## From Scalar Product to Contraction

The outer product asks whether factors add new span. The contraction asks a metric question: if a blade is tested against another blade, what lower-grade piece remains? A vector contracted into a plane returns a vector in that plane, rotated by the plane's orientation and scaled by the vector's metric components in the plane.

This is not merely a symbolic trick. It gives a grade-lowering operation that lets a subspace participate in measurement without first being dissolved into coordinates. The figure below is a movable 3-D object: rotate it, zoom it, and compare the red vector, its blue metric projection, and the green contracted vector inside the plane.


In [4]:
algebra = Algebra([1, 1, 1], names=["e1", "e2", "e3"])
ge1, ge2, ge3 = algebra.basis()
B = ge1.wedge(ge2)
x = 1.2 * ge1 + 0.55 * ge2 + 0.8 * ge3

left = x.left_contract(B)
right = B.right_contract(x)
projected = left.gp(B.inverse_blade()).grade(1)

contraction_report = {
    "B": repr(B),
    "x": repr(x),
    "x_left_contract_B": repr(left),
    "B_right_contract_x": repr(right),
    "projection_from_contraction": repr(projected),
    "projection_coordinates": projected.coordinates().round(6).tolist(),
}

assert np.allclose(projected.coordinates(), np.array([1.2, 0.55, 0.0]))
assert left.almost_equal(-right)
contraction_report


{'B': 'e1^e2',
 'x': '1.2e1 + 0.55e2 + 0.8e3',
 'x_left_contract_B': '-0.55e1 + 1.2e2',
 'B_right_contract_x': '0.55e1 - 1.2e2',
 'projection_from_contraction': '1.2e1 + 0.55e2',
 'projection_coordinates': [1.2, 0.55, 0.0]}

In [5]:
fig = contraction_figure(angle_degrees=42.0)
contraction_path = save_plotly_html(fig, ARTIFACT_TOPIC, "plots", "contraction-vector-plane.html", root=ARTIFACT_ROOT)
display_artifact(contraction_path, width="100%", height=620)


## Duality: Direct and Orthogonal Descriptions

A direct blade stores the subspace itself. Its dual stores the orthogonal complement, using the pseudoscalar of the whole metric space as the conversion device. In 3-D, the dual of an oriented plane is a normal vector; the dual of a vector is the plane perpendicular to it.

This shift is one of the chapter's main programming lessons. Some operations are clearer with the direct element, while others are shorter with the complement. The algebra lets code switch representation without introducing a separate normal-vector data model.


## Left and Right Contraction in Practice

Left and right contraction are easy to blur together because they agree on ordinary vector dot products. They diverge as soon as grades differ and orientation has room to matter. In this notebook, contracting a vector into the oriented plane `B = e1 wedge e2` gives the vector in the plane that remains after the input vector has been metrically tested against the plane's factors. Reversing the direction of contraction changes the sign in this example. That sign is not decoration; it is the orientation bookkeeping that tells later formulas how to compose.

The projection formula shows why this is useful. The contraction `x left-contract B` is not itself the orthogonal projection of `x` onto the plane. It is the plane-oriented leftover. Multiplying by the inverse of the blade converts that leftover back into the direct projected vector. Written in coordinates, this may look longer than deleting the `e3` component. Written geometrically, it is much better: the same idea works when the plane is tilted, scaled, or represented by a nonorthogonal pair of spanning vectors.

The notebook also keeps a coordinate Gram projection nearby. That is not because coordinates are the preferred explanation, but because independent checks are healthy. If the contraction-based projection and the Gram-solve projection agree, we have evidence that the algebraic formula and the numerical implementation are saying the same thing. If they disagree, the likely culprits are grade selection, a missing reverse, or a sign convention mismatch.

A good experiment is to change the plane basis in the projection cell so it is no longer aligned with the coordinate axes. The residual should remain orthogonal to both basis vectors, even though its coordinate entries are no longer visually obvious. That is the difference between a geometric invariant and a coordinate shortcut.


In [6]:
I3 = algebra.pseudoscalar()
plane = (ge1 + 0.25 * ge2).wedge(ge3)
normal = plane.dual().grade(1)
recovered_plane = normal.dual().grade(2)

duality_report = {
    "pseudoscalar": repr(I3),
    "plane": repr(plane),
    "dual_normal": repr(normal),
    "dual_again": repr(recovered_plane),
    "normal_dot_plane_factor_1": normal.left_contract(ge1 + 0.25 * ge2).scalar_value(),
    "normal_dot_plane_factor_2": normal.left_contract(ge3).scalar_value(),
}

assert abs(duality_report["normal_dot_plane_factor_1"]) < 1e-12
assert abs(duality_report["normal_dot_plane_factor_2"]) < 1e-12
duality_report


{'pseudoscalar': 'e1^e2^e3',
 'plane': 'e1^e3 + 0.25e2^e3',
 'dual_normal': '0.25e1 - e2',
 'dual_again': '-e1^e3 - 0.25e2^e3',
 'normal_dot_plane_factor_1': 0.0,
 'normal_dot_plane_factor_2': 0.0}

## Projection as a Metric Operation

Projection can be written with contractions, and it can also be checked with a Gram solve. The two views should agree: the projected vector lies in the target span, and the residual is metric-orthogonal to every spanning vector of that target.

The figure uses a nonorthogonal basis for the plane so that the notebook cannot accidentally rely on a coordinate-axis shortcut.


## Direct Versus Dual Representation

Direct and dual representations are two descriptions of the same geometric constraint. A direct blade says, "these directions span the subspace." A dual blade says, "these directions are perpendicular to the subspace." In 3-D, a plane and its normal vector are the familiar case, but the idea is more general. A line in 3-D has a dual plane of normals. A hyperplane in higher dimensions has a dual vector. The pseudoscalar is the bridge because it represents the oriented full space in which complementarity is being measured.

This is why duality depends on the metric and the ambient space. The word perpendicular has no meaning until a metric is chosen, and the complement of a subspace depends on the dimension of the surrounding vector space. A vector in 2-D dualizes to another vector rotated by a quarter turn; a vector in 3-D dualizes to a bivector plane. The algebra makes these distinctions explicit through grade and pseudoscalar choice.

For code design, this helps avoid a common split-brain model. Traditional geometry libraries often maintain separate types for points, directions, normals, planes, and coordinate frames, with conversion functions scattered among them. A geometric algebra implementation can keep the element as a blade or multivector and choose direct or dual interpretation as needed. That does not eliminate all representation decisions, but it puts the conversions under algebraic control.

Projection and color conversion both benefit from this discipline. Projection uses a blade and its inverse to keep the target subspace central. Reciprocal frames use the metric to construct vectors that are dual to a chosen basis in the coordinate-extraction sense. In both cases, the dual object is not an unrelated helper; it is the metric companion of the original geometric data.


In [7]:
basis = np.array([[1.0, 0.0, 0.0], [0.35, 1.0, 0.0]])
vector = np.array([1.15, 0.85, 1.05])
projection, residual, coeffs = project_vector_to_span(vector, basis)

projection_report = {
    "vector": vector.tolist(),
    "basis": basis.tolist(),
    "projection": projection.round(6).tolist(),
    "residual": residual.round(6).tolist(),
    "coefficients": coeffs.round(6).tolist(),
    "basis_dot_residual": (basis @ residual).round(12).tolist(),
}

assert np.allclose(basis @ residual, np.zeros(2), atol=1e-10)
assert np.allclose(projection + residual, vector)
projection_report


{'vector': [1.15, 0.85, 1.05],
 'basis': [[1.0, 0.0, 0.0], [0.35, 1.0, 0.0]],
 'projection': [1.15, 0.85, 0.0],
 'residual': [0.0, 0.0, 1.05],
 'coefficients': [0.8525, 0.85],
 'basis_dot_residual': [0.0, 0.0]}

In [8]:
fig = projection_figure(vector=vector, basis=basis)
projection_path = save_plotly_html(fig, ARTIFACT_TOPIC, "plots", "orthogonal-projection.html", root=ARTIFACT_ROOT)
display_artifact(projection_path, width="100%", height=620)


## The 3-D Cross Product as a Dual Area

The cross product is useful in ordinary 3-D vector algebra, but it hides two choices: it only works in three dimensions, and it silently uses the ambient orientation. Geometric algebra makes those choices visible. First form the oriented area blade `a wedge b`; then dualize it with the 3-D pseudoscalar to get the vector that classical notation calls `a cross b`.

This notebook keeps the cross product as a bridge, not as the central concept. The blade is the actual oriented plane element; the cross-product vector is one dual representation of it.


In [9]:
a_coords = np.array([1.0, 2.0, 0.5])
b_coords = np.array([-0.25, 1.5, 1.0])
a = algebra.vector(a_coords)
b = algebra.vector(b_coords)
area_blade = a.wedge(b)
cross_ga = area_blade.dual().grade(1).coordinates()
cross_np = cross_from_dual_wedge(a_coords, b_coords)

cross_report = {
    "area_blade": repr(area_blade),
    "dual_vector_from_ga": cross_ga.round(6).tolist(),
    "numpy_cross": cross_np.round(6).tolist(),
    "orthogonal_to_a": float(np.dot(cross_ga, a_coords)),
    "orthogonal_to_b": float(np.dot(cross_ga, b_coords)),
    "area_measure": vector_norm(cross_ga),
}

assert np.allclose(cross_ga, cross_np)
assert abs(cross_report["orthogonal_to_a"]) < 1e-12
assert abs(cross_report["orthogonal_to_b"]) < 1e-12
cross_report


{'area_blade': '2e1^e2 + 1.125e1^e3 + 1.25e2^e3',
 'dual_vector_from_ga': [1.25, -1.125, 2.0],
 'numpy_cross': [1.25, -1.125, 2.0],
 'orthogonal_to_a': 0.0,
 'orthogonal_to_b': 0.0,
 'area_measure': 2.613068120045859}

## Reciprocal Frames

A frame does not have to be orthonormal to be useful. If its vectors are independent, there is a reciprocal frame whose vectors answer coordinate questions. The rule is simple: each original frame vector has dot product one with its own reciprocal and zero with the others.

That makes coordinate extraction a metric operation. Instead of solving a custom linear system each time application code needs coordinates, compute the reciprocal frame once and use contractions or dot products against it.


In [10]:
frame = np.array(
    [
        [1.0, 0.15, 0.05],
        [0.25, 1.0, 0.20],
        [0.10, 0.35, 1.0],
    ]
)
reciprocal = reciprocal_frame(frame)
delta = frame @ reciprocal.T
target = np.array([0.35, 0.55, 0.80])
coords = coordinates_in_frame(target, frame)
reconstructed = coords @ frame

reciprocal_report = {
    "frame": frame.round(6).tolist(),
    "reciprocal": reciprocal.round(6).tolist(),
    "frame_dot_reciprocal": delta.round(10).tolist(),
    "target": target.tolist(),
    "coordinates": coords.round(6).tolist(),
    "reconstructed": reconstructed.round(6).tolist(),
}

assert np.allclose(delta, np.eye(3), atol=1e-10)
assert np.allclose(reconstructed, target, atol=1e-10)
reciprocal_report


{'frame': [[1.0, 0.15, 0.05], [0.25, 1.0, 0.2], [0.1, 0.35, 1.0]],
 'reciprocal': [[1.039251, -0.257019, -0.013968],
  [-0.148065, 1.111887, -0.374354],
  [-0.022349, -0.209526, 1.075569]],
 'frame_dot_reciprocal': [[1.0, -0.0, 0.0], [-0.0, 1.0, 0.0], [0.0, 0.0, 1.0]],
 'target': [0.35, 0.55, 0.8],
 'coordinates': [0.211203, 0.260232, 0.737393],
 'reconstructed': [0.35, 0.55, 0.8]}

In [11]:
fig = reciprocal_frame_figure(frame)
reciprocal_path = save_plotly_html(fig, ARTIFACT_TOPIC, "plots", "reciprocal-frame.html", root=ARTIFACT_ROOT)
display_artifact(reciprocal_path, width="100%", height=620)


## Color-Space Lab

RGB triples can be treated as vectors in a three-dimensional color space. If a camera, display, or artistic palette uses skewed primaries, a reciprocal frame converts observed colors into coordinates relative to those primaries.

The lab below builds a small set of color vectors and a hue wheel around the white direction. The left pane shows the source vectors. The right pane shows coordinates computed by the reciprocal frame and clipped into displayable RGB. The colors themselves are used as marker colors, so the plot doubles as a coordinate check and a visual sanity check.


In [12]:
color_frame = np.array(
    [
        [1.0, 0.08, 0.02],
        [0.15, 1.0, 0.10],
        [0.08, 0.18, 1.0],
    ]
)
sample_colors = sample_rgb_points()
color_coords, color_clipped = color_space_convert(sample_colors, color_frame)

color_report = {
    "input_frame": color_frame.round(6).tolist(),
    "reciprocal_frame": reciprocal_frame(color_frame).round(6).tolist(),
    "sample_colors": sample_colors.round(6).tolist(),
    "converted_coordinates": color_coords.round(6).tolist(),
    "clipped_display_colors": color_clipped.round(6).tolist(),
}

assert np.all(np.isfinite(color_coords))
assert np.all((color_clipped >= 0.0) & (color_clipped <= 1.0))
color_report


{'input_frame': [[1.0, 0.08, 0.02], [0.15, 1.0, 0.1], [0.08, 0.18, 1.0]],
 'reciprocal_frame': [[1.01281, -0.146455, -0.054663],
  [-0.078797, 1.029724, -0.179047],
  [-0.012376, -0.100043, 1.018998]],
 'sample_colors': [[1.0, 0.0, 0.0],
  [0.0, 1.0, 0.0],
  [0.0, 0.0, 1.0],
  [1.0, 1.0, 0.0],
  [1.0, 0.0, 1.0],
  [0.0, 1.0, 1.0],
  [0.9, 0.45, 0.15],
  [0.25, 0.55, 0.9],
  [0.7, 0.7, 0.7],
  [1.0, 1.0, 1.0]],
 'converted_coordinates': [[1.01281, -0.078797, -0.012376],
  [-0.146455, 1.029724, -0.100043],
  [-0.054663, -0.179047, 1.018998],
  [0.866355, 0.950927, -0.11242],
  [0.958147, -0.257844, 1.006621],
  [-0.201118, 0.850678, 0.918955],
  [0.837424, 0.365602, 0.096691],
  [0.123456, 0.385507, 0.85898],
  [0.568184, 0.540316, 0.634605],
  [0.811692, 0.771881, 0.906578]],
 'clipped_display_colors': [[1.0, 0.0, 0.0],
  [0.0, 1.0, 0.0],
  [0.0, 0.0, 1.0],
  [0.866355, 0.950927, 0.0],
  [0.958147, 0.0, 1.0],
  [0.0, 0.850678, 0.918955],
  [0.837424, 0.365602, 0.096691],
  [0.123456, 0.

In [13]:
fig = color_space_figure(color_frame)
color_path = save_plotly_html(fig, ARTIFACT_TOPIC, "plots", "color-space-reciprocal-frame.html", root=ARTIFACT_ROOT)
display_artifact(color_path, width="100%", height=560)


## Sanity Checks

The assertions above are intentionally small. They are the invariants that should remain true when you change the numbers: bivector area is recovered by the reverse-adjusted metric product, vector-plane contraction gives the expected projection when multiplied by the inverse blade, projection residuals are orthogonal to the target span, cross products match dualized area blades in 3-D, and reciprocal frames reconstruct coordinates.


In [14]:
sanity = {
    "bivector_metric_norm_ok": abs(blade_norm_squared(A) - 1.0) < 1e-12,
    "contraction_projection_ok": np.allclose(projected.coordinates(), np.array([1.2, 0.55, 0.0])),
    "projection_residual_orthogonal_ok": np.allclose(basis @ residual, np.zeros(2), atol=1e-10),
    "cross_is_dual_area_ok": np.allclose(cross_ga, cross_np),
    "reciprocal_delta_ok": np.allclose(delta, np.eye(3), atol=1e-10),
    "color_coordinates_finite_ok": bool(np.all(np.isfinite(color_coords))),
}
assert all(sanity.values())

combined_report = {
    "page_audit": page_audit,
    "metric_report": metric_report,
    "contraction_report": contraction_report,
    "duality_report": duality_report,
    "projection_report": projection_report,
    "cross_report": cross_report,
    "reciprocal_report": reciprocal_report,
    "color_report": color_report,
    "sanity": sanity,
}
check_path = save_json(combined_report, ARTIFACT_TOPIC, "checks", "metric-products-sanity.json", root=ARTIFACT_ROOT)
print(f"wrote {check_path}")
sanity


wrote Geometric-Algebra-for-Computer-Science/artifacts/chapter-03/checks/metric-products-sanity.json


{'bivector_metric_norm_ok': True,
 'contraction_projection_ok': True,
 'projection_residual_orthogonal_ok': True,
 'cross_is_dual_area_ok': True,
 'reciprocal_delta_ok': True,
 'color_coordinates_finite_ok': True}

## Lab Prompts for Further Checking

After the notebook validates once, treat the numbers as invitations rather than fixtures. Replace the vector in the contraction section with a vector that lies entirely in the plane. The rejected part should vanish, and the contraction should still encode the in-plane component with the plane's orientation. Replace it with a vector perpendicular to the plane. The projection should collapse to zero, while the wedge with the plane would represent the full volume spanned by that perpendicular direction and the plane.

In the reciprocal-frame section, make the frame nearly singular by moving two frame vectors close together. The reciprocal vectors should grow large, and the coordinate extraction will become numerically sensitive. That is not a failure of geometric algebra; it is the metric version of an ill-conditioned basis. The benefit of the reciprocal-frame view is that this sensitivity is visible in the geometry: cramped frame vectors require long reciprocal vectors to keep the dot-product delta table intact.

In the color-space lab, adjust the cross-talk entries in `color_frame`. Small off-diagonal entries model primaries that are mostly red, green, and blue with slight leakage. Larger off-diagonal entries create a more skewed palette. The reciprocal frame compensates by producing coordinates that may leave the displayable cube, which is why clipping is a separate step. The geometry computes coordinates in the chosen frame; display hardware then imposes the practical range from zero to one.

These experiments are the real value of the seed notebook. The prose gives the map, but the invariants make the map testable. If a changed example preserves the invariant, it belongs to the same geometric idea. If it breaks the invariant, the failure usually points to the exact place where metric, orientation, grade, or ambient dimension was mishandled.


## What to Carry Forward

Metric products let subspaces do measured work. The scalar product compares same-grade blades, contractions lower grade while preserving metric meaning, duality swaps a subspace with its orthogonal complement, projection becomes a short algebraic expression, and reciprocal frames turn nonorthonormal coordinates into a first-class geometric tool.

The important programming habit is to test geometric invariants rather than coordinate coincidences. When the invariant survives skewed frames, tilted planes, and changed vectors, the code is usually expressing the geometry instead of just matching a diagram.


## Takeaways

- `Chapter 3: Metric Products of Subspaces` is treated as an executable geometric algebra lesson: definitions are paired with concrete representations, artifacts, and checks.
- The chapter's computations make the model choices visible, so algebraic operations can be inspected instead of accepted as black-box notation.
- The saved artifacts and sanity checks are the audit trail for this standalone notebook; they support the text without reproducing textbook prose or figures.